In [0]:
# Read all JSON files from Workspace path using Python native I/O
# (spark.read cannot access /Workspace/ paths on Serverless compute)
import json, glob
from pyspark.sql import DataFrame
from pyspark.sql.types import ArrayType, StructType, MapType
json_files = glob.glob("/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/New Extraction/folder_owner/*.json")

records = []
for f in json_files:
    with open(f) as fh:
        data = json.load(fh)
        # print()
        if isinstance(data.get('entries', []), list):
            records.extend(data.get('entries', []))
        else:
            records.append(data.get('entries', []))

print(f"Found {len(records)} records")

df = spark.createDataFrame(records)
from pyspark.sql import functions as sf

display(
    df.groupBy("SI_NAME").agg(
        sf.array_join(sf.collect_set("SI_OWNER"), " | ").alias("owners_agg"),
        sf.count("SI_ID").alias("si_id_count")
    )
)

In [0]:
%sql
select 
    count(distinct webi_basic.document_id) as total_reports, 
    count(distinct web1.Document_Id) as extracted_reports,
    count(distinct case when aud1.composite_ID is not null then web1.Document_Id end) as accessed_reports
from (select distinct Document_Id,
Document_name,
LaunchPad_folder,
scheduled,
lastUpdate_TS,
Author,
lastUpdata_by,
Number_of_Data_Providers from 
dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_profile
) as web1
left join (select *, composite_ID from dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_basic)  as webi_basic
on trim(web1.Document_Id)=trim(webi_basic.Document_Id)
left join (
    select 
        *, 
        composite_ID
    from dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_audit_history
) as aud1
    on trim(webi_basic.composite_ID)=trim(aud1.composite_ID)
;    



In [0]:
%sql
select *
-- Document_Id, count(distinct variable_id ) as cnt 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage
-- -- 416108
-- group by Document_Id
-- order by cnt desc
where Document_Id='17675421'


In [0]:
%sql
with doc_ids as (
  select explode(array(
    '417186','17675421','417236','415898','417028','17675139','17294606','416108','417159','415251','17688887','415517','416296','17296854','416866','417500','417889','17296846','415147'
  )) as Document_Id
)
select Document_Id, File_location from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
where Document_Id in (select Document_Id from doc_ids)

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage
limit 200

In [0]:
%sql
SELECT DISTINCT Document_Id, Document_name
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
WHERE lower(Document_name) LIKE '%declara%'
   OR lower(Document_name) LIKE '%declaratie%'
   OR lower(Document_name) LIKE '%déclar%'
ORDER BY Document_name

In [0]:
%sql
with full_list as (
  -- Owner Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))

  Union all
  -- Viewing Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
),
scanned_record as (
  select *
from (
  select *,
    row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
  -- where document_id='372638'
)
where rn = 1
),
inscope_list as(
  select distinct Document_Id from full_list
  where 
  Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
--   and 
--   Document_Id not in (select distinct Document_Id from scanned_record where document_name is not null)
), 
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
  on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  where upper(trim(UP1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )

),
Audit_profile as (
    SELECT
    WEBI_Doc_ID,
    COUNT(DISTINCT UserName) AS user_count,
    COUNT(DISTINCT BU) AS bu_count
    FROM Audit_data
    GROUP BY WEBI_Doc_ID
),
total_in_scope as (
  select count(distinct Document_Id) as total_cnt from (
    select distinct Document_Id from full_list
    where Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  )
)
select *
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.bu_count > 1


In [0]:
%sql
with doc_ids as (
  select explode(array(
    '417186','17675421','417236','415898','417028','17675139','17294606','416108','417159','415251','17688887','415517','416296','17296854','416866','417500','417889','17296846','415147'
  )) as Document_Id
),
cms as (
  select 
    Document_Id, 
    Document_name, 
    Full_path, 
    created, 
    lastAuthor
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  where Document_Id in (select Document_Id from doc_ids)
),
creator_bu as (
  select 
    upper(trim(user_ID)) as user_id, 
    BU as creator_BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
),
lastauthor_bu as (
  select 
    upper(trim(user_ID)) as user_id, 
    BU as lastAuthor_BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
),
accessed_bu as (
  select 
    WEBI_Doc_ID, 
    UP.BU as accessed_BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit UA
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles UP
    on upper(trim(UA.UserName)) = upper(trim(UP.user_ID))
  where UA.WEBI_Doc_ID in (select Document_Id from doc_ids)
)
select distinct 
  cms.Document_Id,
  cms.Document_name,
  cms.Full_path,
  coalesce(creator_bu.creator_BU, lastauthor_bu.lastAuthor_BU) as Owner_BU,
  accessed_bu.accessed_BU
from cms
left join creator_bu
  on upper(trim(cms.created)) = creator_bu.user_id
left join lastauthor_bu
  on upper(trim(cms.lastAuthor)) = lastauthor_bu.user_id
left join accessed_bu
  on cms.Document_Id = accessed_bu.WEBI_Doc_ID
where cms.Document_Id in (select Document_Id from doc_ids)

In [0]:
%sql
select distinct cms1.Document_Id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as aud1
on trim(cms1.Document_Id) = trim(aud1.WEBI_Doc_ID)
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full as us1
on upper(trim(us1.uid))=upper(trim(aud1.UserName))
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full as us2
on upper(trim(us2.uid))=upper(trim(coalesce(cms1.lastAuthor, cms1.created)))
where aud1.WEBI_Doc_ID is not null
and cms1.Document_Id not in (
    select distinct document_id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
)
and 
(
upper(Trim(us1.businessUnit)) in (upper('FINPM-FINANCE PROGRAM MANAGEMENT'),upper('GFO-GROUP FINANCE OPERATIONS'),upper('GFC-GROUP FINANCE AND CONTROL'),upper('COFIN-CORPORATE FINANCE & TAX'),upper('Group Finance'),upper('Finance'),upper('Finance & Control')) 
or 
us2.businessUnit in ('FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control')
)

select count(*) from _sqldf

In [0]:
%sql
select distinct cms1.Document_Id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms1
where cms1.Document_Id in (
    select distinct document_id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
)
and cms1.DataSource_Type ='fhsql'
and cms1.SQL_Query in ('Error retrieving Query Plan')

In [0]:
%sql
select DataItem_name, universe_name, DataItem_description
from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
where lower(universe_name) like '%claim%' 


In [0]:
%sql
select count(distinct Document_Id ) from dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_profile

In [0]:
%sql
with file_schedule as (
    select distinct document_id as BO_REPORT_ID,
    document_name as BO_REPORT_NAME, 
    full_path as BO_Repot_launchpad_location,
    case when document_last_author is null then document_created_by
    else document_last_author end as BO_Repot_owner,
    schedule_format as output_format, 
    repetition_type as repeated,
    file_path as output_location
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched
    where destination_type='filesystem'
),
user_profile as(
    select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
    where user_ID is not null
)
select file_schedule.bo_report_id, file_schedule.bo_report_name,  live1.Active_Schedule as still_active, Case when user_profile.full_name is not null then user_profile.full_name else file_schedule.bo_repot_owner end as bo_report_owner, user_profile.BU as business_unit, file_schedule.output_format, file_schedule.repeated, file_schedule.output_location, file_schedule.bo_repot_launchpad_location from file_schedule left join user_profile on upper(trim(file_schedule.BO_Repot_owner))=upper(trim(user_profile.user_ID)) left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as live1 on file_schedule.bo_report_id=live1.document_id

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_data_entries
where Document_Id='370626'